# 01f -- dim_season Seed

**Purpose**: Calendar spine for the new event-sourced facts (ADR-0004). One row
per league season. New facts key `season -> season_id`; `dim_draft_pick.draft_season`
-> `season_id` too.

**Key**: `season_id` (string `"2026-2027"`).

**Horizon (self-extending, changed 2026-07-11)**: always `anchor + 2` future
seasons, where `anchor` is calculated from **today's date** (not
`CFG.draft_year` -- that only sets the league-inception floor for a fresh
build). Re-running this notebook after crossing into a new fantasy season
(Mar 1) automatically spins up the next future season row -- no manual year
bump needed. **History is never dropped**: each run unions the calendar-driven
target range with whatever `season_start_year`s already exist in
`dim_season.parquet`, so seasons that already have draft/ledger data (facts
keyed on `season_id`) stay in the calendar dimension even once they're no
longer "current + N".

**Dates**:
- `season_fantasy_start_date` = Mar 1 of the start year (league year opens).
- `season_fantasy_end_date`   = last day of Feb of the end year.
- `season_nfl_start_date` / `season_nfl_end_date` = public NFL schedule,
  **nullable** until known (left null at seed; backfilled when the schedule drops).

**`relative_nfl_season_number`** (added 2026-07-11): 0 = the season whose
`season_fantasy_start_date`/`season_fantasy_end_date` window contains today
(the anchor); negative = past seasons, positive = future. Recomputed on every
run from the current date, not frozen at first seed -- named for the NFL
season since that's the thing being counted, but anchored via the fantasy
window because `season_nfl_start_date`/`_end_date` are still null until the
real schedule drops. Feeds the dead-money design (current/next/total year --
see PLAN.md).

**Output**: `data/dim_season.parquet`

In [1]:
# ---- Setup & Config (shared module) ----------------------------------------
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, DATA

import pandas as pd
from datetime import date, timedelta

## 1 -- Build

In [2]:
# ---- Build dim_season ------------------------------------------------------
# Self-extending horizon: always anchor + FUTURE_HORIZON seasons ahead, floated
# off the calendar (NOT pinned to CFG.draft_year) -- so re-running after crossing
# into a new fantasy season (Mar 1) automatically spins up the next future row,
# with no manual year bump needed. History is never dropped: a re-run unions the
# calendar-driven target range with whatever season_start_years already exist in
# dim_season.parquet, so seasons that already have draft/ledger data (fact
# tables keyed on season_id) stay in the calendar dimension even once they're
# no longer "current + N".
FUTURE_HORIZON = 2          # always keep this many seasons beyond the anchor
INCEPTION_YEAR = CFG.draft_year  # league's first-ever season (2026); floor for a fresh build

# Seed-time themes (descriptive only; extend as seasons gain identity).
_THEMES = {2026: "Startup Draft (inaugural)"}


def _last_day_of_feb(end_year: int) -> date:
    # Last day of February = (Mar 1) - 1 day; handles leap years automatically.
    return date(end_year, 3, 1) - timedelta(days=1)


def _anchor_start_year(today: date) -> int:
    """Fantasy season "sy" runs Mar 1 sy -> last day of Feb (sy+1). If today
    is on/after Mar 1 of its own calendar year, this year's season has opened
    (anchor = today.year); otherwise we're in Jan/Feb, still inside last
    calendar year's season (anchor = today.year - 1)."""
    return today.year if today >= date(today.year, 3, 1) else today.year - 1


def build_dim_season(inception_year: int, future_horizon: int, existing_path: Path) -> pd.DataFrame:
    anchor_sy = _anchor_start_year(date.today())
    target_max_sy = anchor_sy + future_horizon

    existing_years: set[int] = set()
    if existing_path.exists():
        existing_years = set(pd.read_parquet(existing_path)["season_start_year"].tolist())

    all_years = sorted(existing_years | set(range(inception_year, target_max_sy + 1)))

    rows = []
    for sy in all_years:
        ey = sy + 1
        rows.append({
            "season_id": f"{sy}-{ey}",
            "season_start_year": sy,
            "season_end_year": ey,
            "season_fantasy_start_date": date(sy, 3, 1),
            "season_fantasy_end_date": _last_day_of_feb(ey),
            "season_nfl_start_date": pd.NaT,   # nullable until schedule known
            "season_nfl_end_date": pd.NaT,     # nullable until schedule known
            "theme": _THEMES.get(sy, pd.NA),
        })
    df = pd.DataFrame(rows)
    # datetime64 for the date columns (NaT-friendly, Power BI date type).
    for c in ("season_fantasy_start_date", "season_fantasy_end_date",
              "season_nfl_start_date", "season_nfl_end_date"):
        df[c] = pd.to_datetime(df[c])

    # relative_nfl_season_number: 0 = the anchor season (today's fantasy
    # window); negative = past, positive = future. Recomputed for every row,
    # every run -- so last year's anchor correctly drifts to -1 etc.
    df["relative_nfl_season_number"] = df["season_start_year"] - anchor_sy

    return df


dim_season = build_dim_season(INCEPTION_YEAR, FUTURE_HORIZON, DATA / "dim_season.parquet")
out_path = DATA / "dim_season.parquet"
dim_season.to_parquet(out_path, index=False)
print(f"dim_season: {len(dim_season)} rows -> {out_path}")
dim_season

dim_season: 3 rows -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\dim_season.parquet


,season_id,season_start_year,season_end_year,season_fantasy_start_date,season_fantasy_end_date,season_nfl_start_date,season_nfl_end_date,theme,relative_nfl_season_number
0,2026-2027,2026,2027,2026-03-01,2027-02-28,NaT,NaT,Startup Draft (inaugural),0
1,2027-2028,2027,2028,2027-03-01,2028-02-29,NaT,NaT,NaN,1
2,2028-2029,2028,2029,2028-03-01,2029-02-28,NaT,NaT,NaN,2


## 2 -- Validation

In [3]:
# ---- Validation ------------------------------------------------------------
df = pd.read_parquet(DATA / "dim_season.parquet")
assert df["season_id"].is_unique, "season_id must be unique (PK)"
assert (df["season_end_year"] == df["season_start_year"] + 1).all()
# Fantasy window: Mar 1 -> last day of Feb next year.
assert (df["season_fantasy_start_date"].dt.month == 3).all()
assert (df["season_fantasy_start_date"].dt.day == 1).all()
assert (df["season_fantasy_end_date"].dt.month == 2).all()
# No gaps in the year sequence (history + horizon must be contiguous).
years = df["season_start_year"].sort_values().tolist()
assert years == list(range(years[0], years[-1] + 1)), "season_start_year must be contiguous, no gaps"
# Exactly one anchor row (relative_nfl_season_number == 0), rest offset by year.
assert (df["relative_nfl_season_number"] == 0).sum() == 1, "exactly one anchor season expected"
assert (df["relative_nfl_season_number"] == df["season_start_year"] - df.loc[
    df["relative_nfl_season_number"] == 0, "season_start_year"].iloc[0]).all()
# Always at least FUTURE_HORIZON seasons ahead of the anchor.
assert df["relative_nfl_season_number"].max() >= FUTURE_HORIZON, \
    f"expected at least {FUTURE_HORIZON} future seasons seeded"
print("dim_season OK")
print(df.to_string(index=False))

dim_season OK
season_id  season_start_year  season_end_year season_fantasy_start_date season_fantasy_end_date season_nfl_start_date season_nfl_end_date                     theme  relative_nfl_season_number
2026-2027               2026             2027                2026-03-01              2027-02-28                   NaT                 NaT Startup Draft (inaugural)                           0
2027-2028               2027             2028                2027-03-01              2028-02-29                   NaT                 NaT                       NaN                           1
2028-2029               2028             2029                2028-03-01              2029-02-28                   NaT                 NaT                       NaN                           2
